In [ ]:
# Amazon Q- Claude 4.5
"""
Raster processing with open-source Python libraries
Handles large rasters with parallel processing using Dask
"""
import os
import re
import time
from pathlib import Path
from functools import partial
from multiprocessing import Pool, cpu_count

import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling as ResamplingEnum
import dask.array as da
from dask.diagnostics import ProgressBar
import geopandas as gpd

In [ ]:
# ============================================================================
# CONFIGURATION - Set these variables before running
# ============================================================================

INPUT_DIR = r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING"
OUTPUT_DIR = r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\output"
N_WORKERS = cpu_count()  # Use all cores, or set specific number (e.g., cpu_count() - 2)
CHUNK_SIZE = 2048  # Chunk size for processing large rasters (adjust based on RAM)

# Target projection - use EPSG code
TARGET_CRS = "EPSG:4618"

# SHP to clip data if needed
CLIP_SHP = None  # e.g., r"path\to\clip_shapefile.shp"

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def get_year(filename):
    """Extract 4-digit year from filename"""
    match = re.search(r"(\d{4})", filename)
    return match.group(1) if match else ""


def get_raster_files(directory, pattern="*.tif"):
    """Get list of raster files in directory"""
    rasters = sorted(Path(directory).glob(pattern))
    
    if not rasters:
        print(f"No rasters found in directory: {directory}")
        return []
    
    print(f"Found {len(rasters)} rasters")
    return rasters


# ============================================================================
# PROCESSING FUNCTIONS
# ============================================================================

def reproject_raster(input_path, output_dir, target_crs=TARGET_CRS, clip_shp=None):
    """
    Reproject raster to target CRS, optionally clip with shapefile
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        target_crs: Target CRS (EPSG code as string, e.g., "EPSG:102033")
        clip_shp: Path to shapefile for clipping (optional)
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_path = output_dir / f"{input_path.stem}_P.tif"
        
        print(f"Reprojecting {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            # Get bounds for reprojection
            if clip_shp:
                # Read shapefile and reproject to target CRS
                gdf = gpd.read_file(clip_shp).to_crs(target_crs)
                geoms = [mapping(geom) for geom in gdf.geometry]
                
                # Calculate transform based on clipped bounds
                bounds = gdf.total_bounds
                transform, width, height = calculate_default_transform(
                    src.crs, target_crs, src.width, src.height, *src.bounds
                )
            else:
                geoms = None
                transform, width, height = calculate_default_transform(
                    src.crs, target_crs, src.width, src.height, *src.bounds
                )
            
            kwargs = src.meta.copy()
            kwargs.update({
                'crs': target_crs,
                'transform': transform,
                'width': width,
                'height': height,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=target_crs,
                        resampling=Resampling.bilinear
                    )
        
        # Clip if shapefile provided
        if clip_shp:
            temp_path = output_path.with_suffix('.tmp.tif')
            output_path.rename(temp_path)
            
            with rasterio.open(temp_path) as src:
                out_image, out_transform = mask(src, geoms, crop=True, filled=True, nodata=0)
                
                kwargs = src.meta.copy()
                kwargs.update({
                    'height': out_image.shape[1],
                    'width': out_image.shape[2],
                    'transform': out_transform,
                    'nodata': None
                })
                
                with rasterio.open(output_path, 'w', **kwargs) as dst:
                    dst.write(out_image)
            
            temp_path.unlink()
        
        print(f"Successfully reprojected: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reprojecting {input_path}: {e}")
        return None


def reclassify_binary(input_path, output_dir, remap={0: 1, 1: 2}, nodata_value=1):
    """
    Reclassify raster values: 0->1, 1->2, NoData->1
    Uses Dask for memory-efficient processing of large rasters
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        remap: Dictionary mapping old values to new values
        nodata_value: Value to assign to NoData pixels
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_path = output_dir / f"{input_path.stem}_rc.tif"
        
        print(f"Reclassifying {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            # Read as dask array for memory efficiency
            data = da.from_array(src.read(1), chunks=(CHUNK_SIZE, CHUNK_SIZE))
            
            # Handle NoData
            nodata = src.nodata
            if nodata is not None:
                mask = data == nodata
            else:
                mask = da.isnan(data)
            
            # Reclassify
            result = da.zeros_like(data, dtype=np.uint8)
            for old_val, new_val in remap.items():
                result = da.where(data == old_val, new_val, result)
            
            # Set NoData pixels
            result = da.where(mask, nodata_value, result)
            
            # Compute result
            with ProgressBar():
                result_computed = result.compute()
            
            # Write output
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'uint8',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result_computed, 1)
        
        print(f"Successfully reclassified: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reclassifying {input_path}: {e}")
        return None


# ============================================================================
# BATCH PROCESSING
# ============================================================================

def process_rasters_parallel(process_func, input_files, n_workers=N_WORKERS, **kwargs):
    """
    Process multiple rasters in parallel
    
    Args:
        process_func: Function to apply to each raster
        input_files: List of input file paths
        n_workers: Number of parallel workers
        **kwargs: Additional arguments for process_func
    
    Returns:
        List of output paths
    """
    if not input_files:
        return []
    
    print(f"Processing with {n_workers} workers...")
    
    func = partial(process_func, **kwargs)
    
    with Pool(processes=n_workers) as pool:
        outputs = pool.map(func, input_files)
    
    success_count = sum(1 for p in outputs if p)
    print(f"Process complete: {success_count}/{len(input_files)} succeeded")
    
    return outputs


In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main processing workflow"""
    start_time = time.time()
    
    # Get input files
    input_files = get_raster_files(INPUT_DIR, "*.tif")
    
    if not input_files:
        return
    
    # Step 1: Reproject
    print("\n=== STEP 1: REPROJECTING RASTERS ===")
    reprojected = process_rasters_parallel(
        reproject_raster,
        input_files,
        output_dir=OUTPUT_DIR
    )
    
    # Step 2: Reclassify to binary 
    print("\n=== STEP 2: RECLASSIFYING RASTERS ===")
    reclassified = process_rasters_parallel(
        reclassify_binary,
        input_files,
        output_dir=OUTPUT_DIR
    )
    
    # Report execution time
    elapsed_time = time.time() - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    print(f"\nTotal execution time: {int(minutes)} minutes {seconds:.2f} seconds")


if __name__ == "__main__":
    main()
